# Dr. Venu AI — Lip-Sync Demo (Free Colab Prototype)

This notebook uses the supplied Dr. Venu video as the face/background and creates a **male system-generated voice**, then lip-syncs the video to that voice with Wav2Lip.

**Goal:** produce `dr_venu_ai_lipsync.mp4` for the family demonstration.

The open-source Wav2Lip release is for personal/research/non-commercial use. For a later public/commercial GARRF deployment, use a properly licensed commercial lip-sync service/model.

## 1. Upload / place the video

The ZIP includes `avatar_source_30s.mp4`. Upload it to Colab when prompted, or use the included file if you copied the ZIP contents into `/content`.

In [ ]:
!pip -q install edge-tts gdown
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


In [ ]:
import os, shutil
video = next((x for x in os.listdir('/content') if x.lower().endswith('.mp4')), None)
assert video, 'Please upload avatar_source_30s.mp4'
print('Using video:', video)


## 2. Create a male system voice

This uses Microsoft Edge TTS with an Indian-English male voice (`en-IN-PrabhatNeural`). It is **not your cloned voice**; it is simply a male system voice for today's demo.

In [ ]:
import asyncio, edge_tts

TEXT = '''Hello, I am Dr. Venu AI, your digital educational mentor from GARRF. Welcome. You can ask me questions about education, research, technology, and our projects. Let us learn together.'''

async def make_voice():
    communicate = edge_tts.Communicate(TEXT, 'en-IN-PrabhatNeural', rate='-5%', pitch='0Hz')
    await communicate.save('/content/male_voice.mp3')

await make_voice()
print('Created /content/male_voice.mp3')


In [ ]:
from IPython.display import Audio, display
display(Audio('/content/male_voice.mp3'))


## 3. Install Wav2Lip

Wav2Lip can lip-sync a video to arbitrary target audio. The official project documents the `inference.py --face ... --audio ...` workflow.

In [ ]:
!git clone -q https://github.com/Rudrabha/Wav2Lip.git
%cd /content/Wav2Lip
!pip -q install -r requirements.txt
!mkdir -p checkpoints face_detection/detection/sfd results
!wget -q 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O face_detection/detection/sfd/s3fd.pth
!gdown -q '15G3U08c8xsCkOqQxE38Z2XXDnPcOptNk' -O checkpoints/wav2lip_gan.pth
print('Wav2Lip and checkpoints prepared.')


## 4. Generate the lip-synced video

The padding below gives the model a little more room around the lower face/chin, which can help with this framing.

In [ ]:
%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face /content/{video} --audio /content/male_voice.mp3 --outfile /content/dr_venu_ai_lipsync.mp4 --pads 0 20 0 0 --nosmooth


In [ ]:
from IPython.display import Video, display
display(Video('/content/dr_venu_ai_lipsync.mp4', embed=True, width=720))


In [ ]:
from google.colab import files
files.download('/content/dr_venu_ai_lipsync.mp4')
